In [ ]:
# Full name
NAME = ""
# Institutional email (hm.edu or hmtm.de)
EMAIL = ""

<a href="https://colab.research.google.com/github/aica-wavelab/aica-assignments/blob/main/A2_generation/9_6_ml_ffn_markov.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Melody Generation with Machine Learning

+ **AI in Culture and Arts - Tech Crash Course**
+ **Date:** 21.05.2024
+ **Author:** Dr. Benedikt Zönnchen

In the following we will create music sheets and sound. For those tasks ``Python`` requires external programs that you should install if you are working locally:

1. [Musescore](https://musescore.org/de) (for generating sheets)
2. [FluidSynth](https://www.fluidsynth.org/) (for generating sound)

If you are working on google ``Colab``, you can evaluate the following to cells to install these applications:

In [ ]:
#@title install dependencies to play sound
%%capture
print('installing fluidsynth...')
!apt-get install fluidsynth > /dev/null
!cp /usr/share/sounds/sf2/FluidR3_GM.sf2 ./font.sf2
print('done!')

In [ ]:
#@title install dependencies to show score in music notation
%%capture
print('installing musescore3...')
!apt-get install musescore3 > /dev/null
print('done!')

Furtheremore, for this notebook we need the following ``Python`` packages and moduls. Execute the cell to install them:

In [ ]:
#@title Setup: install required Python packages

%pip install music21
%pip install pyfluidsynth

%pip install matplotlib
%pip install seaborn

%pip install pandas
%pip install numpy
%pip install torch

%pip install otter-grader==5.5.0

In [ ]:
#@title Setup: download assignment files (run this cell)

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # download test files
    import os, requests

    folders = ['data', 'figs']
    link = "https://api.github.com/repos/aica-wavelab/aica-assignments/contents/A2_generation"

    def download(entry, dest):
        if entry.get('type') != 'file' or not entry.get('download_url'):
            return
        r = requests.get(entry['download_url'])
        r.raise_for_status()
        with open(dest, 'wb') as out:
            out.write(r.content)

    for folder in folders:
        os.makedirs(folder, exist_ok=True)
        for f in requests.get(f"{link}/{folder}").json():
            download(f, f"{folder}/{f['name']}")

    for f in requests.get(link).json():
        if f['name'].endswith('.py'):
            download(f, f['name'])

## 23 Learning the Markov Matrix

### Generative Models

In this section, we will develop our first **generative model** using ``torch`` (PyTorch). We usually call a model *generative* if its input and output are of the same form. For example *ChatGPT* is generative because we put text in and get text out. Less obvious are diffusion models that generate images but the process also start by putting an image in to get an image out.

### Mathematical Assumptions: Markov Chains

Mathematically, we assume a Markov chain. It is a mathematical model for stochastic processes describing a sequence of possible events. Importantly the model assumes that the probability of each event depends only on the state attained in the previous event which is a strong assumption. It is often refered as the Markov property and sometimes characterized as memorylessness.

Formally, a Markov chain is a sequence of random variables $X_1, X_2, \ldots, X_k$ with the Markov property, namely that the probability of moving to the next state depends only on the present and not on the previous states:

\begin{equation*}
P(X_{k+1} = x_{k+1} | X_k = x_k, X_{k-1} = x_{k-1} \ldots, X_1 = x_1) = P(X_{k+1} = x_{k+1} | X_k = x_k).
\end{equation*}

In our context this means that the next musical event, e.g. the next note, only depends on the previous one. This is, of course, not the case since composers establish much more complicated / interesting relations!

So what we are trying to learn is the probability distribution $P$ assuming the Markov condition.

### Our Machine Learning Model

As we will see, the model consists of a single square matrix $\mathbf{W} \in \mathbb{R}^{n \times n}$ that contains *all* of the **learnable** parameters — that is, the entries of $\mathbf{W}$ change during *training*. 

Given a *one-hot encoded* note/event $\mathbf{x} \in \mathbb{R}^{n}$, we predict the next note/event by computing

$$\mathbf{x}^{\top} \mathbf{W}.$$

Since $\mathbf{x}$ is one-hot — say with its $1$ in position $j$ — this operation simply selects the $j$-th row of $\mathbf{W}$. That row is what the model uses to score the possible next events. In fact, each row in \mathbf{W} stores the (unnormalized) **log-probabilities** — also called *logits* — of the next event given the current one. To turn a row into a proper probability distribution we apply the *softmax*:

$$p_{ij} = \frac{\exp(W_{ij})}{\sum_{k} \exp(W_{ik})}.$$

As we will see, after training the row turns out to be (up to an additive constant) the *logarithm* of a discrete probability distribution: applying $\exp$ element-wise and normalizing each row to sum to $1$ recovers a matrix very similar to our Markov matrix $\mathbf{M}$.

### Why log-probabilities?

The reason becomes clear once we look at the *loss function*. Suppose that, for one training example, the model assigns probability $p$ to the correct next event. We want $p$ to be as large as possible, so a reasonable loss for this single example is $-p$: smaller loss corresponds to a higher predicted probability of the correct label.

For an entire training set with correct-label probabilities $p_1, \ldots, p_m$, the joint probability — assuming the examples are independent — is

$$\text{LH} = p_1 \cdot p_2 \cdot \ldots \cdot p_m,$$

which is called the **likelihood** of the data under the model, i.e. how likely it is that our model produces the observed data. We want to *maximize* $\text{LH}$, or equivalently *minimize* $-\text{LH}$, since we want a high probability that our model produces the training data.

However, multiplying many small probabilities is numerically unstable, and the resulting gradients are awkward. We can avoid both problems by working with the logarithm. Because $\log$ is monotonically increasing, maximizing $\text{LH}$ is the same as maximizing $\log \text{LH}$, and minimizing $-\text{LH}$ is the same as minimizing the **negative log-likelihood** (NLL). The log turns the product into a sum:

$$\text{NLL} = -\log \text{LH} = -\log(p_1 \cdot \ldots \cdot p_m) = -\big(\log p_1 + \ldots + \log p_m\big).$$

This is much friendlier both numerically and analytically. And because the loss is expressed in terms of $\log p_i$, it is natural to have the model predict $\log p_i$ directly rather than $p_i$ itself. 

We will train this model on the same data we used to estimate the Markov matrix $\mathbf{M}$, and we will see that — perhaps surprisingly — after training, the row-normalized $\exp(\mathbf{W})$ and $\mathbf{M}$ are strikingly similar.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import music21 as m21
from music21.stream import Part, Score

import seaborn as sns

from random import choice,shuffle

from pianoroll import stream_to_df, plot_df
from encoder import NoteToIntEncoder, TERM_SYMBOL

In [ ]:
# Let us load our simple melody
bach_minuet = m21.converter.parse('data/Minuet_in_G.mid')
bach_minuet_melody = bach_minuet.parts[0]
bach_minuet_melody.show('midi')

We can display the melody to get a in a *piano roll representation* to get a feel for it.

In [ ]:
dataframe = stream_to_df(bach_minuet_melody)
dataframe.head()

In [ ]:
plot_df(dataframe)

### 23.1 Data Preparation

Similar to our Markov example, we transform the ``Stream`` into a list of numbers ranging from $0$ to $n-1$.

In [ ]:
# initiate our encoder
encoder = NoteToIntEncoder([bach_minuet_melody])

# encode the Stream into a squence of numbers
enc_bach_minuet = encoder.encode_sequence(bach_minuet_melody)

# add terminal symbol to indicate the begining and end of the piece
enc_bach_minuet = [encoder.encode(TERM_SYMBOL)] + enc_bach_minuet + [encoder.encode(TERM_SYMBOL)]

print(enc_bach_minuet)

Next we specify some (hyper-)parameters. Note that the learning rate is unusually large.

In [ ]:
# Parameters
vocab_size = len(encoder) # all MIDI keys
batch_size = 32
learning_rate = 1.0
epochs = 500

print(f'vocab_size: {vocab_size}')

Next we have to generate our training data set. Each row of the input matrix $\mathbf{X}$ represents one hot-encoded note/event while the (output/labels) vector $\mathbf{y}$ is not one-hot encoded.

In [ ]:
import torch
import torch.nn.functional as F

# generate training data
X = []
y = []
for i in range(len(enc_bach_minuet) - 1):
    current_event = enc_bach_minuet[i]
    next_event = enc_bach_minuet[i + 1]
    X.append(current_event)
    y.append(next_event)

X = F.one_hot(torch.tensor(X, dtype=torch.long), num_classes=vocab_size).float()
y = torch.tensor(y, dtype=torch.long)

print(X[:3])
print(y[:3])

The **forward pass** is computed in two steps. First, we multiply the input matrix by the weight matrix:

$$\mathbf{C} = \mathbf{X} \cdot \mathbf{W},$$

with $\mathbf{W} \in \mathbb{R}^{n \times n}$, $\mathbf{X} \in \mathbb{R}^{m \times n}$, and therefore $\mathbf{C} \in \mathbb{R}^{m \times n}$. $\mathbf{C}$ consits of the *logits* and since each row of $\mathbf{X}$ is a one-hot encoded musical event / note, $\mathbf{C}$ just consits of rows of $\mathbf{W}$.

Next, we apply the **softmax** operation **row-wise**:

$$p_{ki} = \frac{\exp(c_{ki})}{\sum_{j=1}^{n} \exp(c_{kj})} \quad \text{for } k = 1, \ldots, m, \; i = 1, \ldots, n.$$

In other words, we apply the exponential function element-wise and then **normalize** each row to sum to $1$. Each row of the resulting matrix $\mathbf{P}$ can therefore be interpreted as a discrete probability distribution over the $n$ possible next events.

To optimize $\mathbf{W}$ via *gradient descent* we need a *loss function* $L$. We start from the *likelihood* of the data: let $p_1, \ldots, p_m$ be the probabilities the model assigns to the correct next event for each of the $m$ transitions (one per row of $\mathbf{X}$). The likelihood is then

$$\text{LH} = p_1 \cdot p_2 \cdot \ldots \cdot p_m.$$

We want to *maximize* $\text{LH}$, but products of many small probabilities are numerically unstable. Taking the logarithm turns the product into a sum, and flipping the sign turns "maximize" into "minimize," giving us the **negative log-likelihood**:

$$L = -\frac{1}{m}\big(\log p_1 + \log p_2 + \ldots + \log p_m\big).$$

This is the loss function we minimize. Note that we introduced a factor of $1/m$ such that the loss magnitude doesn't grow with the dataset size.

Calling ``loss.backward()`` performs **backpropagation** (the *backward pass*), which computes the gradient $\nabla_{\mathbf{W}} L$. We then update the weights with a standard gradient-descent step:

$$\mathbf{W} \leftarrow \mathbf{W} - \eta \cdot \nabla_{\mathbf{W}} L.$$

In our case, we choose a comparatively **large learning rate** $\eta$. This is unusual for deep-learning models, but justified here: our model has only a single linear layer and no nonlinearity beyond the softmax, so the loss surface is well-behaved and large steps converge much faster without overshooting.

### 23.2 Model Definition and Training

In [ ]:
import torch

# Initialize weights
W = torch.randn(vocab_size, vocab_size, dtype=torch.float32, requires_grad=True)

# Training loop
epochs = 1000
learning_rate = 10.0

for k in range(epochs):
    # 1. Forward pass
    C = X @ W                               # m x n x n x n => m x n
    C = torch.exp(C)                        # component-wise exp
    P = C / C.sum(dim=1, keepdim=True)      # sum over columns (dim = 1) to get one number for each row! => P in m x n

    # Compute loss
    # (1) torch.arange(len(y)) produces a 1-D tensor [0, 1, 2, ..., m-1] => 1 x m.
    #
    # (2) P[torch.arange(len(y)), y] is advanced (fancy) indexing. 
    # When you pass two 1-D index tensors of the same length to a 2-D tensor, 
    # PyTorch pairs them up element-by-element rather than taking a Cartesian product. 
    # So this expression returns
    # [P[0,y0​],P[1,y1​],P[2,y2​],...,P[m−1,ym−1​]],
    # i.e., for each row k, it picks the column corresponding to the correct label y_k
    #
    # (3) torch.log(...) takes the natural log of each of those probabilities elementwise
    #
    # (4) -...mean() averages them and flips the sign
    loss = -torch.log(P[torch.arange(len(y)), y]).mean()

    if k % 100 == 0:
        print(f'Epoch {k}, Loss: {loss.item()}')

    # 2. Backward pass
    if W.grad is not None:
        W.grad.zero_()
    loss.backward()

    # 3. Update weights
    with torch.no_grad():
        W -= learning_rate * W.grad

### 23.3 Result Analysis

Let us now compre the matrix $\mathbf{W}$ with our Markov matrix $\mathbf{M}$ from the previous section.
Let's recompute $\mathbf{M}$.

In [ ]:
def markov_matrix(parts, encoder):
  m = len(encoder)

  # initiate the Markov matrix with zeros
  M = np.zeros((m,m))

  # count the transitions; a row represents the starting point and the column the end point
  for part in parts:
    predecessor = TERM_SYMBOL
    idx2 = 0
    for element in part.recurse():
      if isinstance(element, (m21.note.Rest, m21.note.Note)):
        idx1 = encoder.encode(predecessor)
        idx2 = encoder.encode(element)
        M[idx1][idx2] += 1.0
        predecessor = element
      # this is for the ending since we arrive at a TERM_SYMBOL
    M[idx2][encoder.encode(TERM_SYMBOL)] += 1.0

  # divide each element of a row by the sum of that row
  return M / M.sum(axis=1, keepdims=True)

In [ ]:
M = markov_matrix([bach_minuet_melody], encoder)

Let us now transform $\mathbf{W}$ into a matrix representing probabilities and let us round the entries:

In [ ]:
P = torch.exp(W)
P = P / P.sum(dim=1, keepdim=True)
P = torch.round(P, decimals=1)

Let us plot both matrices:

In [ ]:
sns.heatmap(P.detach().numpy(), cmap="Blues")

In [ ]:
sns.heatmap(M, cmap="Blues")

### 4.1.4 Melody Generation

<!-- BEGIN QUESTION -->

---

🖍 **Exercise 23.1:** The following function ``ffn_score(W, encoder, max_len)`` generates a new ``Score`` based on our computed matrix ``W``. It works similar to ``markov_score(M, encoder, max_len)``.

Explain in your own words, what is going on. Try to find out what each operation does. Try to play around with toy examples. For example, you could generate a ``numpy`` matrix to test what ``P = C / np.sum(C, axis=1, keepdims=True)`` compute.

---

In [ ]:
def ffn_score(W, encoder, max_len=100):
  score = Score()
  part = Part()

  C = np.exp(W)
  P = C / np.sum(C, axis=1, keepdims=True)
  
  j = encoder.encode(TERM_SYMBOL)
  for _ in range(max_len):
    m = len(encoder)
    j = np.random.choice(m, size=1, p=P[j])[0]
    symbol = encoder.decode(j)
    if symbol == TERM_SYMBOL:
      break
    else: 
      part.append(symbol)
  score.insert(0, part)
  return score

ffn_score(W.detach().numpy(), encoder).show('midi')

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

---

🖍 **Exercise 23.1:** As we saw, using just one matrix to be the neural network--which is in fact the most simplest network we can imagine--we basically learn the Markov matrix.

1. Is this method more or less computational expensive compared to our computation of the Markov matrix?
2. What do we have to change to consider not only the last note but the last two notes to compute the next one?

---

_Type your answer here, replacing this text._

<!-- END QUESTION -->

